In [5]:
import subprocess
import re
import pandas as pd
import shutil
from tqdm import tqdm
import openpyxl
import os
import gc

## Diretorios

In [6]:
quartus_bin = r"C:\altera_standard\25.1std\quartus\bin64"
base_dir = rf"C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_trunc"

m_values = [16, 32, 64, 128, 256, 512]


# m_values = [256]
# n_values = [256] 

## Functions

In [7]:
# =========================
# altera n no VHDL
# =========================
def set_n(vhdl_file, n):
    with open(vhdl_file, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    text = re.sub(
        r"n\s*:\s*integer\s*:=\s*\d+",
        f"n : integer := {n}",
        text
    )

    with open(vhdl_file, "w", encoding="utf-8") as f:
        f.write(text)


# =========================
# acha nome do projeto .qpf
# =========================
def find_project_name(project_dir):
    qpf_files = [f for f in os.listdir(project_dir) if f.endswith(".qpf")]

    if not qpf_files:
        raise FileNotFoundError(f"Nenhum arquivo .qpf encontrado em {project_dir}")

    return os.path.splitext(qpf_files[0])[0]


# =========================
# roda quartus
# =========================
def run_quartus(project_dir):
    shutil.rmtree(os.path.join(project_dir, "db"), ignore_errors=True)
    shutil.rmtree(os.path.join(project_dir, "incremental_db"), ignore_errors=True)

    project_name = find_project_name(project_dir)

    try:
        subprocess.run(
            [
                os.path.join(quartus_bin, "quartus_sh.exe"),
                "--flow",
                "compile",
                project_name
            ],
            cwd=project_dir,
            check=True,
            capture_output=True,
            text=True
        )

        subprocess.run(
            [
                os.path.join(quartus_bin, "quartus_sta.exe"),
                "-t",
                "script.tcl"
            ],
            cwd=project_dir,
            check=True,
            capture_output=True,
            text=True
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            f"Quartus failed in {project_dir} with return code {e.returncode}\n"
            f"stdout:\n{e.stdout}\n"
            f"stderr:\n{e.stderr}"
        ) from e


# =========================
# parse dos paths
# =========================
def parse_paths(file):
    paths = []
    current = None

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:

            m = re.search(r"Path #\d+: Delay is\s+([\d\.]+)", line)
            if m:
                if current:
                    paths.append(current)

                current = {
                    "delay": float(m.group(1)),
                    "ic": None,
                    "cell": None
                }
                continue

            if current is None:
                continue

            m = re.search(r";\s*IC\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                current["ic"] = float(m.group(1))

            m = re.search(r";\s*Cell\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                current["cell"] = float(m.group(1))

    if current:
        paths.append(current)

    df = pd.DataFrame(paths)

    if df.empty:
        return df

    return df.dropna()


# =========================
# calcula métricas
# =========================
def compute_metrics(df):
    critical = df.loc[df["delay"].idxmax()]

    return {
        "Delay_CP": critical["delay"],
        "IC_CP": critical["ic"],
        "CELL_CP": critical["cell"],

        "Delay_MAX": df["delay"].max(),
        "IC_MAX": df["ic"].max(),
        "CELL_MAX": df["cell"].max(),

        "Delay_MEAN": df["delay"].mean(),
        "IC_MEAN": df["ic"].mean(),
        "CELL_MEAN": df["cell"].mean()
    }


# =========================
# extrai caminho crítico
# =========================
def extract_critical_path(file):
    delay = None
    ic = None
    cell = None
    from_node = None
    to_node = None

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:

            m = re.search(r"Delay is\s+([\d\.]+)", line)
            if m:
                delay = float(m.group(1))

            m = re.search(r";\s*From Node\s*;\s*(.*?)\s*;", line)
            if m:
                from_node = m.group(1)

            m = re.search(r";\s*To Node\s*;\s*(.*?)\s*;", line)
            if m:
                to_node = m.group(1)

            m = re.search(r";\s*IC\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                ic = float(m.group(1))

            m = re.search(r";\s*Cell\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                cell = float(m.group(1))

    return {
        "delay": delay,
        "ic": ic,
        "cell": cell,
        "from": from_node,
        "to": to_node
    }


In [8]:
# =========================
# loop principal
# =========================
resultados = []

for num_vec in tqdm(m_values):

    project_dir = os.path.join(base_dir, f"v{num_vec}")
    vhdl_file = os.path.join(project_dir, "vhdl", f"sum{num_vec}xn.vhd")
    report_file = os.path.join(project_dir, "io_paths.txt")
    worst_file = os.path.join(project_dir, "worst_path.txt")

    resultados_m = []

    print(f"\n==============================")
    print(f" Rodando projeto com {num_vec} vetores")
    print(f" Casos: {num_vec}x{num_vec} e {num_vec}x{2*num_vec}")
    print(f"==============================")

    # roda N e 2N
    for n in [num_vec, 2*num_vec]:

        df = None
        metrics = None
        cp = None

        try:
            print(f"\nRodando caso {num_vec}x{n}")

            set_n(vhdl_file, n)
            run_quartus(project_dir)

            df = parse_paths(report_file)

            if df.empty:
                raise ValueError("Nenhum path encontrado em io_paths.txt")

            metrics = compute_metrics(df)
            cp = extract_critical_path(worst_file)

            linha = {
                "num_vec": num_vec,
                "n": n,
                "caso": f"{num_vec}x{n}",

                "Delay_CP": cp["delay"],
                "IC_CP": cp["ic"],
                "CELL_CP": cp["cell"],

                "From_Node": cp["from"],
                "To_Node": cp["to"],

                "Delay_MAX": metrics["Delay_MAX"],
                "IC_MAX": metrics["IC_MAX"],
                "CELL_MAX": metrics["CELL_MAX"],

                "Delay_MEAN": metrics["Delay_MEAN"],
                "IC_MEAN": metrics["IC_MEAN"],
                "CELL_MEAN": metrics["CELL_MEAN"],
            }

            resultados.append(linha)
            resultados_m.append(linha)

        except Exception as e:
            print(f"Erro em num_vec={num_vec}, n={n}: {e}")
            continue

        finally:
            for name in ("df", "metrics", "cp"):
                globals().pop(name, None)
            gc.collect()

    # salva um arquivo separado para cada número de vetores
    if resultados_m:
        df_m = pd.DataFrame(resultados_m).sort_values("n")

        csv_m = os.path.join(base_dir, f"timing_v{num_vec}_N_2N.csv")
        xlsx_m = os.path.join(base_dir, f"timing_v{num_vec}_N_2N.xlsx")

        df_m.to_csv(csv_m, index=False)
        df_m.to_excel(xlsx_m, index=False)

        print(f"\nArquivos salvos para {num_vec} vetores:")
        print(csv_m)
        print(xlsx_m)


# =========================
# salva resultado geral
# =========================
df_all = pd.DataFrame(resultados).sort_values(["num_vec", "n"])

csv_all = os.path.join(base_dir, "timing_N_2N.csv")
xlsx_all = os.path.join(base_dir, "timing_N_2N.xlsx")

df_all.to_csv(csv_all, index=False)
df_all.to_excel(xlsx_all, index=False)

print("\nArquivos gerais salvos:")
print(csv_all)
print(xlsx_all)

  0%|          | 0/6 [00:00<?, ?it/s]


 Rodando projeto com 16 vetores
 Casos: 16x16 e 16x32

Rodando caso 16x16

Rodando caso 16x32


 17%|█▋        | 1/6 [04:08<20:43, 248.62s/it]


Arquivos salvos para 16 vetores:
C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_trunc\timing_v16_N_2N.csv
C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_trunc\timing_v16_N_2N.xlsx

 Rodando projeto com 32 vetores
 Casos: 32x32 e 32x64

Rodando caso 32x32

Rodando caso 32x64


 33%|███▎      | 2/6 [08:21<16:44, 251.11s/it]

...


Arquivos salvos para 128 vetores:
C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_trunc\timing_v128_N_2N.csv
C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_trunc\timing_v128_N_2N.xlsx

 Rodando projeto com 256 vetores
 Casos: 256x256 e 256x512

Rodando caso 256x256

Rodando caso 256x512


 83%|████████▎ | 5/6 [1:02:34<18:34, 1114.06s/it]


Arquivos salvos para 256 vetores:
C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_trunc\timing_v256_N_2N.csv
C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_trunc\timing_v256_N_2N.xlsx

 Rodando projeto com 512 vetores
 Casos: 512x512 e 512x1024

Rodando caso 512x512

Rodando caso 512x1024


: 

: 

In [ ]:
# =========================

# salva resultado geral
# =========================
final_df = pd.DataFrame(resultados).sort_values(["num_vec", "n"])

csv_final = os.path.join(base_dir, "timing_all_vectors.csv")
xlsx_final = os.path.join(base_dir, "timing_all_vectors.xlsx")

final_df.to_csv(csv_final, index=False)
final_df.to_excel(xlsx_final, index=False)

print("\nResultado final:")
print(final_df)

print("\nArquivos gerais salvos:")
print(csv_final)
print(xlsx_final)

NameError: name 'pd' is not defined